# 去重版 Stage1 0bp 失败归因

只读已保存的 B/C 原生实物账户，按相同 SOXX 基准分解现金、SOXL 选择、费用和残差；不回放策略。

In [1]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.analysis.etf_transfer_attribution import holdings_attribution
from src.analysis.integrated_allocation import paired_metrics
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_common_return_dedup_0bp_failure_attribution_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
parent=root/src['candidate_card'];original=json.loads((parent/'analysis_result.json').read_text());binding=json.loads((parent/'nas_binding.json').read_text());assert sha(parent/'plan.json')==src['candidate_plan_sha256'] and sha(parent/'analysis_result.json')==src['candidate_analysis_result_sha256'] and binding['evidence']['sha256']==src['candidate_NAS_sha256']
assert original['candidate_SOXX_relative_wealth']<1 and original['new_native_accounts']==2 and original['source_Recorder_status_after_repair']=='FINISHED'
assert sha(root/'src/analysis/etf_transfer_attribution.py')==src['helper_sha256']
source_card=root/src['matched_source_card'];assert sha(source_card/'summary.json')==src['matched_source_summary_sha256']
legacy=root/src['original_Stage1_component_card']/'component_result.json';assert sha(legacy)==src['original_Stage1_component_result_sha256'];A=json.loads(legacy.read_text())
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['candidate_experiment_id'],recorder_id=src['candidate_recorder_id']);assert rec.status=='FINISHED'
 candidate={}
 for family in ('B_original55','C_common_dedup'):
  for kind in ('native','targets'):
   path=parent/'qlib_recorder'/src['candidate_experiment_id']/src['candidate_recorder_id']/'artifacts'/kind/family
   if kind=='targets':assert sha(path)==src['saved_'+('B' if family.startswith('B_') else 'C')+'_target_artifact_sha256']
   candidate[(kind,family)]=rec.load_object(kind+'/'+family)
with R.uri_context(str(source_card/'qlib_recorder')):
 summary=json.loads((source_card/'summary.json').read_text());rec=R.get_recorder(experiment_id=summary['experiment_id'],recorder_id=summary['recorder_id']);assert rec.status=='FINISHED'
 frame=rec.load_object('inputs/frame');prices=rec.load_object('inputs/raw_points');schedule=rec.load_object('inputs/schedule')
rawspec=json.loads((root/'research/stage1_domain_heads_20261018/plan.json').read_text())['sources']['raw'];rawcard=root/rawspec['card'];rawsum=json.loads((rawcard/'summary.json').read_text())
with R.uri_context(str(rawcard/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=rawsum['experiment_id'],recorder_id=rawspec['recorder_id']);assert rec.status=='FINISHED';SOXX=rec.load_object('native/raw_SOXX_0')
assert SOXX['audit']['status']=='passed' and len(SOXX['authority'])==944
ref=SOXX['authority'].reset_index(drop=True);fields=['cash_opportunity_log','SOXL_selection_log','direct_fees_log','timing_and_benchmark_residual_log'];daily_frames={};segments_frames={};periods={};rolling={};groups={}
for family in ('B_original55','C_common_dedup'):
 book=candidate[('native',family)];target=candidate[('targets',family)];assert book['audit']['status']=='passed' and len(book['authority'])==944
 source_frame=frame.set_index('signal_date').copy();pd.testing.assert_index_equal(source_frame.index,target.index,check_names=False)
 for name in target:source_frame[name]=target[name].to_numpy(copy=True)
 daily,segments=holdings_attribution(book,prices,schedule,source_frame,initial_cash=100000)
 assert len(daily)==944 and len(segments)==944*2*4 and daily.closure_error.abs().max()<1e-9
 pd.testing.assert_frame_equal(book['authority'][['trade_date','exit_date']].reset_index(drop=True),ref[['trade_date','exit_date']].reset_index(drop=True))
 previous=np.r_[100000.,book['authority'].native_expected_nav.to_numpy(float)[:-1]];post=pd.DatetimeIndex(daily.trade_date)+pd.Timedelta(hours=9,minutes=47);marks=book['points'].set_index('stamp').loc[post]
 w_soxl=marks.q_soxl.to_numpy(float)*prices.loc[post,'SOXL'].to_numpy(float)/previous;w_cash=(marks.cash.to_numpy(float)+marks.receivable.to_numpy(float))/previous
 candidate_return=daily['return'].to_numpy(float);benchmark=ref.strategy_return.to_numpy(float);cash=-w_cash*benchmark;selection=daily.SOXL_contribution.to_numpy(float)-w_soxl*benchmark;fees=daily.fee_contribution.to_numpy(float);relative=candidate_return-benchmark;residual=relative-cash-selection-fees;paired=np.log1p(candidate_return)-np.log1p(benchmark);scale=np.divide(paired,relative,out=1/(1+benchmark),where=np.abs(relative)>1e-12)
 comp=pd.DataFrame({'signal_date':book['authority'].signal_date,'trade_date':daily.trade_date,'exit_date':daily.exit_date,'paired_log_excess':paired,'post_open_cash_weight':w_cash,'post_open_soxl_weight':w_soxl,'cash_opportunity_log':cash*scale,'SOXL_selection_log':selection*scale,'direct_fees_log':fees*scale,'timing_and_benchmark_residual_log':residual*scale,'risk_state':daily.risk_state})
 assert np.max(np.abs(comp[fields].sum(axis=1)-comp.paired_log_excess))<1e-12
 assert abs(float(comp.paired_log_excess.sum())-paired_metrics(book['authority'],ref)['log_excess_sum'])<1e-9
 daily_frames[family]=comp;segments_frames[family]=segments
 def describe(group,label):return {'period':label,'sessions':len(group),'relative_log_excess':float(group.paired_log_excess.sum()),'relative_wealth':float(np.exp(group.paired_log_excess.sum())),'mean_post_open_cash_weight':float(group.post_open_cash_weight.mean()),'mean_post_open_soxl_weight':float(group.post_open_soxl_weight.mean()),**{k:float(group[k].sum()) for k in fields}}
 all_rows=[describe(comp,'all')]
 for frequency in ('Y','Q'):
  for period,g in comp.groupby(comp.trade_date.dt.to_period(frequency),sort=True):all_rows.append(describe(g,str(period)))
 periods[family]=pd.DataFrame(all_rows)
 windows=pd.DataFrame([{'first_trade':str(comp.trade_date.iloc[i].date()),'last_exit':str(comp.exit_date.iloc[i+125].date()),**describe(comp.iloc[i:i+126],'rolling126')} for i in range(len(comp)-125)])
 assert len(windows)==819;rolling[family]=windows
 nonwinning=windows.relative_log_excess.le(0);groups[family]={'nonwinning_126_windows':int(nonwinning.sum()),'nonwinning_cash_negative':int((nonwinning&windows.cash_opportunity_log.lt(0)).sum()),'nonwinning_selection_negative':int((nonwinning&windows.SOXL_selection_log.lt(0)).sum())}
full={family:{k:float(periods[family].iloc[0][k]) for k in ['relative_log_excess','relative_wealth','mean_post_open_cash_weight','mean_post_open_soxl_weight',*fields]} for family in periods}
full_delta={k:full['C_common_dedup'][k]-full['B_original55'][k] for k in fields}
C2025=periods['C_common_dedup'].loc[periods['C_common_dedup'].period.eq('2025')].iloc[0]
result={'decision':'saved_native_common_return_dedup_failure_attributed_no_policy','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['candidate_recorder_id'],'source_NAS_sha256':src['candidate_NAS_sha256'],'sessions':944,'quarters':16,'rolling126_windows':819,'full':full,'C_minus_B_full_log_component_differences_descriptive':full_delta,'C_2025':{k:float(C2025[k]) for k in ['relative_log_excess','relative_wealth',*fields]},'original_A_full_log_components':A['full_log_components'],'nonwinning_window_components':groups,'component_closure_max_abs':{family:float(np.max(np.abs(daily_frames[family][fields].sum(axis=1)-daily_frames[family].paired_log_excess))) for family in daily_frames},'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan)
 rec.save_objects(artifact_path='analysis',daily=daily_frames,segments=segments_frames,periods=periods,rolling126=rolling,result=result)
 rec.log_metrics(source_sessions=944,rolling126_windows=819,model_fits=0,native_accounts=0)
 rid,eid=rec.id,str(rec.experiment_id)
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
result.update(recorder_id=rid,experiment_id=eid,recorder_uri=str(card/'qlib_recorder'),NAS_sha256=remote,NAS_remote_hash_verified=True);(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k not in ('original_A_full_log_components',)},ensure_ascii=False,indent=2))


{
  "decision": "saved_native_common_return_dedup_failure_attributed_no_policy",
  "plan_sha256": "1f7140824c1e02ae5a611afe49238ae1f4fbeb22c499c5aae0225706b1c0fb0e",
  "source_recorder_id": "a42f11bdd1484bde86d4dbb1e4855af3",
  "source_NAS_sha256": "cd98ce324cb0c19b393bafc149ad727c6c55bd7860d8f915eae3cf2d43c0f86d",
  "sessions": 944,
  "quarters": 16,
  "rolling126_windows": 819,
  "full": {
    "B_original55": {
      "relative_log_excess": -0.3593918264390208,
      "relative_wealth": 0.698100763419373,
      "mean_post_open_cash_weight": 0.4889700529939973,
      "mean_post_open_soxl_weight": 0.14571368611322008,
      "cash_opportunity_log": -0.6108293291660948,
      "SOXL_selection_log": 0.25342074627578154,
      "direct_fees_log": -0.010300652992589567,
      "timing_and_benchmark_residual_log": 0.00831740944388196
    },
    "C_common_dedup": {
      "relative_log_excess": -0.03928917033085769,
      "relative_wealth": 0.9614726395822718,
      "mean_post_open_cash_weight": 0.

## 中文结论

B/C两本已保存账户均逐日闭合。C全程现金机会−0.40256、SOXL选择+0.36432；2025现金−0.10556被选择+0.04034部分抵消；C未赢450个滚动窗口里418个现金为负。去重实质改善但不满足SOXX硬门，固定版本关闭。0新增拟合或账户，Recorder/NAS冷读一致。